# 249 - Statistics at the peak-variance K

**Run this only after 240, 241 and 242 have ALL finished.** It reads their three sweep files, picks the K where convex NMF's held-out variance peaks - separately for each feature set - and runs the full battery at that K for all three methods.

| | |
|---|---|
| nulls per cell | 50 |
| LOPO null reps | 10 |
| statistics | separation vs matched null, cross-method agreement, anatomical coherence, leave-one-patient-out |


In [1]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd(); sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'functions'))
BSF = ROOT/'outputs'/'clustering'/'bsf_comparison'
FEATURE_SETS = ['concat_hg', 'concat_rawds']
N_NULL, LOPO_REPS = 50, 10

def sh(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, *[str(c) for c in cmd]], cwd=str(ROOT),
                       env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
    assert r.returncode == 0, f'failed: {cmd}'
print('nulls per cell:', N_NULL, ' LOPO null reps:', LOPO_REPS)

nulls per cell: 50  LOPO null reps: 10


## 1 - Merge the three sweeps and pick K

The peak is read from **convex NMF** because it is the only method whose bi-cross-validated curve turns over; k-means and Ward rise monotonically, so their argmax is just the largest K tested and means nothing. That same K is then used for all three.

In [2]:
# The three sweeps wrote one file each. Merge them, then take the K where
# convex NMF's held-out variance PEAKS - per feature set, independently.
parts = []
for m in ('kmeans','hierarchical','cnmf'):
    f = BSF/f'heldout_variance_{m}.csv'
    assert f.exists(), f'{f.name} missing - has 24x finished?'
    parts.append(pd.read_csv(f))
hv = pd.concat(parts, ignore_index=True)
hv.to_csv(BSF/'heldout_variance_ALL.csv', index=False)

g = (hv[hv.scheme=='home'].groupby(['feature_set','method_label','k'])['var_explained']
     .mean().reset_index())
PEAK_K = {}
for fs in FEATURE_SETS:
    c = g[(g.feature_set==fs) & (g.method_label=='convex NMF')].sort_values('k')
    PEAK_K[fs] = int(c.loc[c.var_explained.idxmax(),'k'])
    edge = PEAK_K[fs] in (c.k.min(), c.k.max())
    print(f'{fs:<14} cNMF peak K = {PEAK_K[fs]}   '
          f'var_explained {c.var_explained.max():.4f}'
          + ('   <-- AT THE EDGE OF THE RANGE, the curve did not turn over' if edge else ''))
json.dump(PEAK_K, open(BSF/'peak_k.json','w'), indent=2)
PEAK_K

concat_hg      cNMF peak K = 11   var_explained 0.5519
concat_rawds   cNMF peak K = 12   var_explained 0.4627


{'concat_hg': 11, 'concat_rawds': 12}

## 2 - The statistics, at that K

In [3]:
# Same K for all three methods, per feature set. K may differ BETWEEN feature
# sets - that is expected and fine, they are different representations.
for fs in FEATURE_SETS:
    sh(['make_cluster_statistics.py', '--feature-set', fs, '--k', PEAK_K[fs],
        '--n-null', N_NULL, '--lopo-reps', LOPO_REPS])

$ make_cluster_statistics.py --feature-set concat_hg --k 11 --n-null 50 --lopo-reps 10
$ make_cluster_statistics.py --feature-set concat_rawds --k 12 --n-null 50 --lopo-reps 10


## 3 - Figures

In [5]:
# C.3a/b/c and C.8a/b/c at the peak K, per feature set, plus the K=5..30 curve.
for fs in FEATURE_SETS:
    sh(['make_cluster_figures.py', '--feature-set', fs, '--k', PEAK_K[fs]])
sh(['make_heldout_figure.py'])

from IPython.display import Image, display as disp
for fs in FEATURE_SETS:
    d_ = ROOT/'outputs'/'clustering'/'statistics'/f'{fs}_K{PEAK_K[fs]}'
    for png in sorted(d_.glob('C3*.png')) + sorted(d_.glob('C8*.png')):
        print(png.name); disp(Image(str(png), width=1100))

$ make_cluster_figures.py --feature-set concat_hg --k 11


AssertionError: failed: ['make_cluster_figures.py', '--feature-set', 'concat_hg', '--k', 11]

## 4 - Read the numbers

In [6]:
for fs in FEATURE_SETS:
    d = ROOT/'outputs'/'clustering'/'statistics'/f'{fs}_K{PEAK_K[fs]}'
    print(f'=== {fs}  K={PEAK_K[fs]} ===')
    for f in ('stats_separation.csv','stats_agreement.csv','stats_coherence.csv'):
        p = d/f
        if p.exists():
            print(f'--- {f} ---'); display(pd.read_csv(p))

=== concat_hg  K=11 ===
--- stats_separation.csv ---


,method,method_label,space,home,silhouette,null_mean,null_sd,z,n_null
0,kmeans,k-means (BSF),dB,True,0.069142,0.065859,0.001978,1.659963,50
1,kmeans,k-means (BSF),unit-norm,False,-0.069325,0.058914,0.002554,-50.210179,50
2,hierarchical,Ward,dB,True,0.061281,0.034415,0.003227,8.326014,50
3,hierarchical,Ward,unit-norm,False,-0.105798,0.031309,0.005304,-25.852152,50
4,cnmf,convex NMF,dB,False,0.018421,0.041794,0.005128,-4.558140,50
5,cnmf,convex NMF,unit-norm,True,0.046582,0.035092,0.003966,2.897156,50


--- stats_agreement.csv ---


,a,a_label,b,b_label,ari,nmi
0,kmeans,k-means (BSF),hierarchical,Ward,0.292916,0.473242
1,kmeans,k-means (BSF),cnmf,convex NMF,0.149366,0.305905
2,hierarchical,Ward,cnmf,convex NMF,0.128260,0.289261


--- stats_coherence.csv ---


...

,method,method_label,space,home,silhouette,null_mean,null_sd,z,n_null
0,kmeans,k-means (BSF),dB,True,0.068801,0.038442,0.001005,30.210555,50
1,kmeans,k-means (BSF),unit-norm,False,0.054445,0.033064,0.001717,12.449105,50
2,hierarchical,Ward,dB,True,0.062759,0.013777,0.001942,25.223698,50
3,hierarchical,Ward,unit-norm,False,0.006627,0.009179,0.003663,-0.696533,50
4,cnmf,convex NMF,dB,False,0.032674,0.017583,0.002459,6.137423,50
5,cnmf,convex NMF,unit-norm,True,0.079337,0.016581,0.001843,34.042969,50


--- stats_agreement.csv ---


,a,a_label,b,b_label,ari,nmi
0,kmeans,k-means (BSF),hierarchical,Ward,0.356785,0.566593
1,kmeans,k-means (BSF),cnmf,convex NMF,0.364289,0.530468
2,hierarchical,Ward,cnmf,convex NMF,0.252950,0.472094


--- stats_coherence.csv ---


,method,method_label,neighbours_sharing_label,over_chance
0,kmeans,k-means (BSF),0.253638,2.471762
1,hierarchical,Ward,0.317332,2.181203
2,cnmf,convex NMF,0.270114,2.773851
